# Lab 03 — Search by Meaning, Not Just Words

A pretrained model may create embeddings. Your job is to implement cosine similarity and deterministic top-*k* retrieval in NumPy.

**Rules:** CPU only; no vector database; no hosted embedding API; no library nearest-neighbour/search implementation.

In [1]:
%pip -q install sentence-transformers numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from time import perf_counter
import numpy as np
from sentence_transformers import SentenceTransformer

np.set_printoptions(precision=4, suppress=True)
RNG = np.random.default_rng(42)
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
experiment_started = perf_counter()

c:\Users\alexm\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Challenge corpus

The data is deterministic and deliberately contains paraphrases. It is lab-only data, not the InstaKill domain model.

In [3]:
TOPICS = {
    "food": [
        "A budget-friendly restaurant opened near the station.",
        "Looking for an affordable place to eat dinner tonight.",
        "The cheapest lunch menu I found this week costs under ten euros.",
    ],
    "running": [
        "Easy jogging route beside the river for beginners.",
        "My first 5K training plan starts with short runs and walks.",
        "Anyone know a calm place for an evening run?",
    ],
    "python": [
        "I finally understood why a Python dictionary needs hashable keys.",
        "Small NumPy experiment: vectorized code was much faster than my loop.",
        "Looking for a clear explanation of Python type hints.",
    ],
    "music": [
        "Quiet jazz playlist for a long study session.",
        "Any recommendations for live music this weekend?",
        "This new album is perfect background sound while reading.",
    ],
    "travel": [
        "Best low-cost train trip for a one-day escape?",
        "I need ideas for a weekend hike without a car.",
        "What should I pack for a short mountain trip in autumn?",
    ],
    "books": [
        "Recommend a short science-fiction novel with a strong ending.",
        "I want a book that explains machine learning without too much math.",
        "Which library has a good quiet corner for reading?",
    ],
}

def make_demo_posts(repetitions=200):
    posts = []
    post_id = 1
    for topic, templates in TOPICS.items():
        for repeat in range(repetitions):
            template = templates[repeat % len(templates)]
            posts.append({
                "post_id": post_id,
                "user_id": (repeat % 37) + 1,
                "topic": topic,
                "content": f"{template} [community note {repeat + 1}]",
            })
            post_id += 1
    RNG.shuffle(posts)
    return posts

posts = make_demo_posts()
texts = [post["content"] for post in posts]
assert len(posts) >= 1_000
print(f"{len(posts)} posts; example: {posts[0]}")

1200 posts; example: {'post_id': 70, 'user_id': 33, 'topic': 'food', 'content': 'A budget-friendly restaurant opened near the station. [community note 70]'}


## Baseline: keyword overlap

This baseline is intentionally simple. You may inspect it, but do not use it as your semantic solution.

In [4]:
def keyword_search(query, posts, k=5):
    query_terms = set(query.lower().split())
    scored = []
    for post in posts:
        score = len(query_terms & set(post["content"].lower().split()))
        scored.append((score, post["post_id"], post))
    scored.sort(key=lambda item: (-item[0], item[1]))
    return [post for score, _, post in scored[:k] if score > 0]

keyword_search("cheap dinner", posts, k=3)

[{'post_id': 2,
  'user_id': 2,
  'topic': 'food',
  'content': 'Looking for an affordable place to eat dinner tonight. [community note 2]'},
 {'post_id': 5,
  'user_id': 5,
  'topic': 'food',
  'content': 'Looking for an affordable place to eat dinner tonight. [community note 5]'},
 {'post_id': 8,
  'user_id': 8,
  'topic': 'food',
  'content': 'Looking for an affordable place to eat dinner tonight. [community note 8]'}]

## TODO 1 — Encode text

Load the supplied pretrained encoder and create a normalized embedding matrix for all corpus texts. Keep the model call at this boundary; your retrieval functions below must not load a model or perform I/O.

In [5]:
model = SentenceTransformer(MODEL_NAME, device="cpu")
corpus_embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(f"Embedding matrix: {corpus_embeddings.shape}, {corpus_embeddings.dtype}")

Batches: 100%|██████████| 38/38 [00:01<00:00, 21.58it/s]

Embedding matrix: (1200, 384), float32


## TODO 2 — Pure retrieval functions

Implement these functions with NumPy. A zero vector must not produce `NaN` or infinity. For equal scores, the lower `post_id` comes first.

In [6]:
def cosine_similarity(query_vector, corpus_vectors):
    """Return one cosine score for every row in corpus_vectors."""
    query = np.asarray(query_vector, dtype=np.float64)
    vectors = np.asarray(corpus_vectors, dtype=np.float64)
    if query.ndim != 1 or vectors.ndim != 2 or vectors.shape[1] != query.shape[0]:
        raise ValueError("Expected a 1-D query and a 2-D matrix with matching dimensions.")
    query_norm = np.linalg.norm(query)
    row_norms = np.linalg.norm(vectors, axis=1)
    denominators = row_norms * query_norm
    scores = np.zeros(vectors.shape[0], dtype=np.float64)
    np.divide(vectors @ query, denominators, out=scores, where=denominators != 0)
    return scores


def stable_top_k(scores, post_ids, k):
    """Return indices ordered by score descending, then post_id ascending."""
    scores = np.asarray(scores, dtype=np.float64)
    post_ids = np.asarray(post_ids)
    if scores.ndim != 1 or post_ids.ndim != 1 or scores.shape != post_ids.shape:
        raise ValueError("Scores and post IDs must be one-dimensional arrays of equal length.")
    if k < 0:
        raise ValueError("k must be non-negative.")
    if not np.isfinite(scores).all():
        raise ValueError("Scores must all be finite.")
    return np.lexsort((post_ids, -scores))[:k]


def semantic_search(query_embedding, posts, corpus_embeddings, k=5):
    """Return up to k dictionaries with post data and a numeric score."""
    scores = cosine_similarity(query_embedding, corpus_embeddings)
    post_ids = np.asarray([post["post_id"] for post in posts])
    indices = stable_top_k(scores, post_ids, k)
    return [posts[int(index)] | {"score": float(scores[index])} for index in indices]

## Acceptance checks

Run this cell only after implementing the TODOs. These checks are deliberately small; they verify the retrieval contract, not whether a model is magically perfect.

In [7]:
tiny_query = np.array([1.0, 0.0])
tiny_corpus = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.0], [0.0, 0.0]])
tiny_scores = cosine_similarity(tiny_query, tiny_corpus)
assert np.allclose(tiny_scores, [1.0, 0.0, 1.0, 0.0])
assert np.isfinite(cosine_similarity(np.array([0.0, 0.0]), tiny_corpus)).all()

tie_indices = stable_top_k(np.array([0.7, 0.9, 0.9]), np.array([7, 5, 2]), k=2)
assert list(tie_indices) == [2, 1], "ties must use ascending post_id"

assert corpus_embeddings.shape[0] == len(posts)
assert np.issubdtype(corpus_embeddings.dtype, np.floating)

query_embedding = model.encode(["cheap dinner near the station"], normalize_embeddings=True)[0]
results = semantic_search(query_embedding, posts, corpus_embeddings, k=5)
assert 1 <= len(results) <= 5
assert all({"post_id", "user_id", "content", "score"} <= set(result) for result in results)
assert all(results[i]["score"] >= results[i + 1]["score"] for i in range(len(results) - 1))
assert all(np.isfinite(result["score"]) for result in results)
print("Acceptance checks passed.")

Acceptance checks passed.


## Compare search behavior

Use at least three queries. For each, compare the keyword baseline and your semantic search. Record one useful result and one surprising or weak result.

In [8]:
queries = [
    "low-cost supper",
    "a beginner's route for jogging",
    "annotating code with expected types",
]

comparison = {}
for query in queries:
    query_embedding = model.encode([query], normalize_embeddings=True)[0]
    semantic = semantic_search(query_embedding, posts, corpus_embeddings, k=5)
    keyword = keyword_search(query, posts, k=5)
    comparison[query] = {"semantic": semantic, "keyword": keyword}
    print(f"\nQUERY: {query}")
    print("semantic:", [(item["post_id"], round(item["score"], 3), item["content"]) for item in semantic[:3]])
    print("keyword:", [(item["post_id"], item["content"]) for item in keyword[:3]])

model_size_mib = sum(parameter.numel() * parameter.element_size() for parameter in model.parameters()) / (1024 ** 2)
embedding_size_mib = corpus_embeddings.nbytes / (1024 ** 2)
runtime_seconds = perf_counter() - experiment_started
print(f"\nCPU runtime (model load, encoding, and comparisons): {runtime_seconds:.1f}s")
print(f"Model parameter size: {model_size_mib:.1f} MiB; corpus embeddings: {embedding_size_mib:.1f} MiB")


QUERY: low-cost supper
semantic: [(177, 0.583, 'The cheapest lunch menu I found this week costs under ten euros. [community note 177]'), (156, 0.577, 'The cheapest lunch menu I found this week costs under ten euros. [community note 156]'), (168, 0.577, 'The cheapest lunch menu I found this week costs under ten euros. [community note 168]')]
keyword: [(801, 'Best low-cost train trip for a one-day escape? [community note 1]'), (804, 'Best low-cost train trip for a one-day escape? [community note 4]'), (807, 'Best low-cost train trip for a one-day escape? [community note 7]')]

QUERY: a beginner's route for jogging
semantic: [(381, 0.744, 'Easy jogging route beside the river for beginners. [community note 181]'), (363, 0.74, 'Easy jogging route beside the river for beginners. [community note 163]'), (366, 0.739, 'Easy jogging route beside the river for beginners. [community note 166]')]
keyword: [(201, 'Easy jogging route beside the river for beginners. [community note 1]'), (204, 'Easy 

### Comparison notes

The useful result is the `low-cost supper` query: semantic search retrieves a post about the cheapest lunch menu even though none of its content words occur in the query. The keyword baseline instead matches `low-cost` in an unrelated train-trip post, showing that token overlap can confuse a shared phrase with shared intent. A weaker behavior is that repeated corpus templates fill several top positions with near-duplicate posts; semantic relevance does not provide result diversity.

## RESULTS

The notes below summarize the measured CPU run and the three query comparisons above.

```text
RESULTS
baseline: Token-overlap search; it matches 'low-cost' in an unrelated train-trip post for the low-cost supper query.
solution: CPU all-MiniLM-L6-v2 embeddings with hand-written NumPy cosine similarity and deterministic top-k (score descending, post_id ascending).
runtime: about 2 seconds on CPU for model loading, corpus encoding, and three queries after the model and dataset were cached; initial downloads excluded. Exact timing is printed by the comparison cell.
memory / model size: 86.6 MiB of model parameters and 1.8 MiB for the 1,200 corpus embeddings; peak process RAM is not measured.
3 failure cases: repeated synthetic templates dominate top-k with near-duplicate posts; keyword overlap can return an unrelated result for a shared phrase; semantic similarity alone may miss exact constraints such as price, time, or location.
what I would try next: Evaluate on real human-written posts with relevance judgments, then report precision@k for a broader set of paraphrased queries.
```